# RiskLens — one project, five competency milestones

**Build Custom AI · Saras AI Institute · Criterion Set v2.0**

Build and evaluate your visual inspector, evidence engine, trained Analyst, optimization comparison and integrated system. Submit this same continuing notebook at five milestones. There are 24 Met / Not Yet criteria across C1–C5; every criterion is required. No point totals, averages or bonus credit apply.

| Module | Evidence | Competency / review first |
|---|---|---|
| 1 | D1.1–D1.7 | C1 / C1.1 frozen image split |
| 2 | D2.1–D2.7 | C2 / C2.3 grounded answers |
| 3 | D3.1–D3.7 | C3 / C3.2 separated evaluation |
| 4 | D4.1–D4.4 | C4 / C4.2 predeclared gate |
| 5 | D4.5–D4.7 + final delivery | C5 / C5.1 real integration |

Use the [full rubric](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone/blob/codex/competency-colab-refresh/RUBRICS.md) and [submission guide](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone/blob/codex/competency-colab-refresh/docs/SUBMISSION.md). Save a personal copy and use a GPU runtime. The setup installs the reviewed package stack while preserving Colab's PyTorch/CUDA pairing.

Fill learner workspaces; preserve GIVEN interfaces and D labels. Unimplemented tasks intentionally raise `NotImplementedError`: this starter is not a finished solution. The final completed notebook must run/load top to bottom using saved weights. Keep real outputs, dated snapshots and accessible artifacts; cell position alone is not evidence of freezing. Technical helpers never assign a competency judgment.

The starter targets a 16 GB T4 with a resident VLM and an Analyst of at most 1.5B parameters, normally using a 4-bit serving route. Actual capacity and availability vary. Optional extensions do not compensate for required evidence.


# 0 · Shared setup and GIVEN interfaces

Run setup first in a fresh session. Section 0 supplies infrastructure; the deliberate `inspect_v0` and `retrieve_v0` weaknesses are retained for learner improvement.


In [ ]:
"""Run this cell first in a fresh Colab session, before importing model libraries."""
import importlib.metadata as _metadata
import platform as _platform
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
from pathlib import Path as _Path

COURSE_PACKAGES = [
    "transformers==5.22.0", "trl==1.17.0", "peft==0.21.2",
    "accelerate==1.15.0", "datasets==5.1.0", "sentence-transformers==6.2.0",
    "faiss-cpu==1.15.1", "num2words==0.5.14",
    "pillow==13.0.0", "scikit-learn==1.9.1", "matplotlib==3.11.2", "requests==2.34.2",
]
if _sys.version_info < (3, 11):
    raise RuntimeError("Use a current Colab Python 3.11+ runtime.")
try:
    _torch_version = _metadata.version("torch")
except _metadata.PackageNotFoundError as _error:
    raise RuntimeError("Use Colab's preinstalled matched PyTorch/torchvision runtime; see docs/COLAB.md.") from _error
_constraints = ["torch==" + _torch_version]
try:
    _constraints.append("torchvision==" + _metadata.version("torchvision"))
except _metadata.PackageNotFoundError:
    raise RuntimeError("The vision labs need a matching torchvision installation; use a fresh Colab runtime.")
if _platform.system() == "Linux":
    COURSE_PACKAGES.append("bitsandbytes==0.50.2")
_before = {}
for _name in ["transformers", "trl", "peft", "datasets", "sentence_transformers", "numpy"]:
    if _name in _sys.modules:
        _before[_name] = getattr(_sys.modules[_name], "__version__", None)
with _tempfile.TemporaryDirectory() as _temporary:
    _constraint_file = _Path(_temporary) / "preserve-colab-torch.txt"
    _constraint_file.write_text("\n".join(_constraints))
    _subprocess.run([_sys.executable, "-m", "pip", "install", "--quiet", "--upgrade-strategy", "only-if-needed",
                     "-c", str(_constraint_file), *COURSE_PACKAGES], check=True)
if _before:
    raise RuntimeError("Packages installed. Restart the Colab session once, then run from the setup cell again before importing models.")
print("Course stack installed; existing PyTorch/torchvision CUDA wheels preserved.")
print("If installation failed, stop here. Start a fresh GPU runtime and inspect the resolver message.")


### Runtime check

Stop if installation reports an error. If packages were already imported, restart the session once and begin again. Record the actual environment in your submission; use saved artifacts for final delivery.


In [ ]:
import torch, json, os, re, time, random
import numpy as np
from importlib.metadata import version

TEAM = "your-name-or-team"  # Use the identifier requested by your course.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU runtime for the complete capstone.")
BF16_OK = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
GPU_PRICE_PER_H = None  # Enter your explicit assumed/provider hourly price; 0 is allowed if justified.
environment = {"device": torch.cuda.get_device_name(0), "dtype": str(DTYPE), "seed": SEED,
               "versions": {n: version(n) for n in ["torch", "torchvision", "transformers", "peft", "trl", "bitsandbytes"]}}
print(json.dumps(environment, indent=2))


### GIVEN · Evidence, contracts and measurement helpers

Embedded from `risklens_support.py`; the saved notebook runs without cloning the repository. Keep the content-hash record plus an independently dated notebook snapshot. Automated evidence checks do not determine Met.


In [ ]:
"""Given evidence utilities for RiskLens. These helpers do not grade competencies.

This module is also embedded in the starter so a saved Colab copy is standalone.
Only standard-library imports belong here; model execution remains in the notebook.
"""

import hashlib
import json
import math
import re
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

REQUIRED_SECTIONS = [
    "## ROOT CAUSE ANALYSIS", "**Product:**", "**Defect:**", "**Severity:**",
    "**Evidence:**", "**Probable cause:**", "**Recommended action:**",
]
REFUSAL_PHRASES = [
    "don't have enough information", "do not have enough information",
    "not enough information", "cannot answer",
]
METRICS_FILE = "capstone_metrics.json"


def stable_hash(value):
    """Hash JSON data; NaN and infinity are never valid evidence values."""
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False)
    return hashlib.sha256(payload.encode()).hexdigest()


def write_json(path, value):
    """Replace a JSON file atomically after serializability validation."""
    text = json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False)
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(target)


def freeze_record(path, payload):
    """Create once or verify identical content; never silently rewrite a frozen plan.

    A local timestamp supports record keeping, not independent proof of chronology.
    Preserve a dated notebook snapshot or version history as required by the rubric.
    """
    target = Path(path)
    digest = stable_hash(payload)
    if target.exists():
        record = json.loads(target.read_text())
        if record.get("sha256") != digest or stable_hash(record["payload"]) != digest:
            raise ValueError(f"Frozen record differs: {target}. Preserve it and use a new version.")
        return record
    record = {"created_utc": datetime.now(timezone.utc).isoformat(),
              "sha256": digest, "payload": payload}
    write_json(target, record)
    return record


def record_metrics(stage, **metrics):
    """Keep a latest view and an append-only JSONL history, including judge/count strings."""
    if not isinstance(stage, str) or not stage.strip():
        raise ValueError("A nonempty stage key is required")
    stable_hash(metrics)  # Validate before touching either file.
    path = Path(METRICS_FILE)
    previous = json.loads(path.read_text()) if path.exists() else {}
    entry = {"stage": stage, "recorded_utc": datetime.now(timezone.utc).isoformat(),
             "metrics": metrics}
    history = path.with_name(path.stem + "_history.jsonl")
    history.parent.mkdir(parents=True, exist_ok=True)
    with history.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(entry, ensure_ascii=False, allow_nan=False) + "\n")
    previous[stage] = metrics
    write_json(path, previous)
    print(f"Recorded {stage}; retain the ledger history and full experiment outputs.")


def show_ledger():
    path = Path(METRICS_FILE)
    print(path.read_text() if path.exists() else "No measured stage has been recorded yet.")


def make_input(defect_note, evidence):
    if not isinstance(defect_note, str) or not isinstance(evidence, str):
        raise TypeError("The Analyst expects a prose note and evidence string, not a dict.")
    return (f"Defect note from visual inspection:\n{defect_note}\n\n"
            f"Retrieved evidence:\n{evidence}\n\nWrite a root cause analysis report.")


def split_input(row):
    text = row["input"]
    prefix = "Defect note from visual inspection:\n"
    separator = "\n\nRetrieved evidence:\n"
    suffix = "\n\nWrite a root cause analysis report."
    if not isinstance(text, str) or not text.startswith(prefix) or not text.endswith(suffix):
        raise ValueError("Input does not follow make_input()")
    body = text[len(prefix):-len(suffix)]
    if separator not in body:
        raise ValueError("Missing evidence boundary")
    return tuple(body.split(separator, 1))


def format_ok(text):
    """Mechanical format only: all headers and one exact LOW/MEDIUM/HIGH severity line."""
    if not isinstance(text, str) or not all(h in text for h in REQUIRED_SECTIONS):
        return False
    lines = re.findall(r"(?m)^\*\*Severity:\*\*[^\n]*$", text)
    return len(lines) == 1 and bool(re.fullmatch(
        r"\*\*Severity:\*\*[ \t]*(LOW|MEDIUM|HIGH)[ \t]*", lines[0]))


def image_id(image):
    """Pixel identity independent of a temporary Colab/cache file path."""
    rgb = image.convert("RGB")
    h = hashlib.sha256(f"RGB:{rgb.size}".encode())
    h.update(rgb.tobytes())
    return h.hexdigest()


def validate_image_split(test_images, dev_images):
    """Check counts, duplicate pixels and overlap; timing still needs version evidence."""
    def describe(rows):
        ids, counts = [], {"good": 0, "defective": 0}
        for image, label in rows:
            if label not in counts:
                raise ValueError(f"Unknown image label {label!r}")
            ids.append(image_id(image)); counts[label] += 1
        if len(ids) != len(set(ids)):
            raise ValueError("Duplicate image pixels within a split")
        return ids, counts
    test_ids, test_counts = describe(test_images)
    dev_ids, dev_counts = describe(dev_images)
    if len(test_ids) < 40 or min(test_counts.values()) < 20:
        raise ValueError("C1.1 needs at least 40 test images and 20 per class")
    if not dev_ids or set(test_ids) & set(dev_ids):
        raise ValueError("A nonempty separate development set is required")
    return {"test_ids": test_ids, "dev_ids": dev_ids,
            "test_counts": test_counts, "dev_counts": dev_counts}


def inspection_metrics(records):
    """Defective is positive. Invalid predictions stay in accuracy and the 2x3 matrix."""
    if not records:
        raise ValueError("Cannot evaluate an empty image set")
    for row in records:
        if row["truth"] not in {"good", "defective"}:
            raise ValueError("Unknown truth label")
        if row["prediction"] not in {"good", "defective", "invalid"}:
            raise ValueError("Unknown prediction label")
    tp = sum(r["truth"] == "defective" and r["prediction"] == "defective" for r in records)
    fp = sum(r["truth"] == "good" and r["prediction"] == "defective" for r in records)
    fn = sum(r["truth"] == "defective" and r["prediction"] != "defective" for r in records)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {"n": len(records), "accuracy": sum(r["truth"] == r["prediction"] for r in records) / len(records),
            "defect_precision": precision, "defect_recall": recall,
            "defect_f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
            "invalid_count": sum(r["prediction"] == "invalid" for r in records),
            "confusion_rows_good_defective_columns_good_defective_invalid": [
                [sum(r["truth"] == t and r["prediction"] == p for r in records)
                 for p in ["good", "defective", "invalid"]] for t in ["good", "defective"]]}


def validate_corpus_gold(corpus, gold, official_docs):
    if len({d["source"] for d in corpus}) != len(corpus):
        raise ValueError("Corpus source IDs must be unique before chunking")
    by_id = {d["source"]: d for d in corpus}
    if sum(d.get("kind") == "review" for d in corpus) < 100:
        raise ValueError("C2.1 requires at least 100 real reviews")
    for doc in official_docs:
        if doc["source"] not in by_id or by_id[doc["source"]]["text"] != doc["text"]:
            raise ValueError("Official CAPSTONE_DOCS must be present verbatim")
    if sum(d.get("kind") == "doc" and d["source"] not in {o["source"] for o in official_docs}
           for d in corpus) < 2:
        raise ValueError("At least two authored documents are required")
    if len(gold) < 20 or len({g["q"] for g in gold}) != len(gold):
        raise ValueError("At least 20 distinct gold questions are required")
    for g in gold:
        if not g.get("sources") or not set(g["sources"]) <= set(by_id) or not g.get("ref"):
            raise ValueError("Every gold question needs valid source IDs and a reference")
        if g.get("multihop") and len(set(g["sources"])) < 2:
            raise ValueError("A multi-hop question requires at least two source IDs")
    if sum(bool(g.get("multihop")) for g in gold) < 3:
        raise ValueError("At least three multi-hop questions are required")
    return {"n_corpus": len(corpus), "n_gold": len(gold),
            "manual_checks": ["real review provenance", "authored facts and link", "multi-hop meaning"]}


def recall_table(retrieve_fn, gold):
    """RiskLens recall@k = fraction of questions with ALL required sources retrieved."""
    if not gold or any(not g.get("sources") for g in gold):
        raise ValueError("Nonempty gold questions and required-source sets are needed")
    table, records = {}, []
    for k in [1, 2, 4, 8]:
        successes = 0
        for g in gold:
            hits = retrieve_fn(g["q"], k)
            if len(hits) > k:
                raise ValueError("Retriever returned more than k hits")
            found = {h["source"] for h in hits}
            complete = set(g["sources"]) <= found
            successes += complete
            records.append({"q": g["q"], "k": k, "required_sources": g["sources"],
                            "hits": hits, "all_required_found": complete})
        table[f"recall@{k}"] = successes / len(gold)
    return table, records


def refusal_diagnostic(answer, hits):
    """Flag phrase/citation syntax; a human must judge support and refusal meaning."""
    if not isinstance(answer, str):
        raise TypeError("Answer must be text")
    cited = re.findall(r"\[([A-Za-z0-9_.:-]+)\]", answer)
    available = {h["source"] for h in hits}
    return {"phrase_match": any(p in answer.lower() for p in REFUSAL_PHRASES),
            "unknown_citations": sorted(set(cited) - available),
            "meaningful_refusal": None, "claim_support": None}


def latency_summary(seconds):
    if not seconds or any(isinstance(x, bool) or not math.isfinite(x) or x <= 0 for x in seconds):
        raise ValueError("Nonempty finite positive durations required")
    ordered = sorted(seconds)
    def percentile(q):
        position = (len(ordered) - 1) * q
        lower = math.floor(position); upper = math.ceil(position)
        return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)
    return {"n_runs": len(seconds), "mean_s": statistics.mean(seconds),
            "p50_s": percentile(.5), "p95_s": percentile(.95), "percentile_method": "linear",
            "sample_limit": "Descriptive sample percentiles, not a production SLA"}


def bench(fn, arg, n_runs=6, warmups=1, synchronize=None):
    """Measure completed calls; pass torch.cuda.synchronize for CUDA workloads."""
    if type(n_runs) is not int or n_runs < 1 or type(warmups) is not int or warmups < 0:
        raise ValueError("Positive run count and nonnegative warm-up count required")
    sync = synchronize or (lambda: None)
    for _ in range(warmups):
        fn(arg); sync()
    times = []
    for _ in range(n_runs):
        sync(); started = time.perf_counter(); fn(arg); sync()
        times.append(time.perf_counter() - started)
    return {**latency_summary(times), "observations_s": times, "warmups": warmups}


def cost_per_1000(mean_seconds, hourly_price, utilization=1.0):
    values = [mean_seconds, hourly_price, utilization]
    if any(isinstance(x, bool) or not isinstance(x, (int, float)) or not math.isfinite(x) for x in values):
        raise ValueError("Finite explicit time, price and utilization are required")
    if mean_seconds < 0 or hourly_price < 0 or not 0 < utilization <= 1:
        raise ValueError("Nonnegative time/price and utilization in (0,1] required")
    return mean_seconds * hourly_price * 1000 / (3600 * utilization)


def quality_gate(reference, candidate, gate):
    """Compare unrounded rates; content uses cosine x100 units, not factual accuracy."""
    values = [reference[k] for k in ["format_rate", "content_cosine"]]
    values += [candidate[k] for k in ["format_rate", "content_cosine"]]
    values += [gate[k] for k in ["max_format_drop_pts", "max_content_drop_pts"]]
    if any(isinstance(x, bool) or not isinstance(x, (float, int)) or not math.isfinite(x) for x in values):
        raise ValueError("All gate measurements and tolerances must be finite numbers")
    if any(not 0 <= arm["format_rate"] <= 1 or not -1 <= arm["content_cosine"] <= 1
           for arm in [reference, candidate]) or min(values[-2:]) < 0:
        raise ValueError("Invalid rate, cosine or negative tolerance")
    fmt = 100 * (reference["format_rate"] - candidate["format_rate"])
    content = 100 * (reference["content_cosine"] - candidate["content_cosine"])
    passed = fmt <= gate["max_format_drop_pts"] + 1e-9 and content <= gate["max_content_drop_pts"] + 1e-9
    return {"format_drop_pts": fmt, "content_drop_pts": content,
            "verdict": "PASS" if passed else "FAIL"}


#### Latest ledger and preserved history


In [ ]:
# Defined in the GIVEN support cell above; this stable cell position is retained.


#### `show_ledger()` prints the latest measured summaries


In [ ]:
# Defined in the GIVEN support cell above; this stable cell position is retained.


### ⚙️ VLM plumbing · *given*

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import model_info

VLM_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
VLM_REVISION = model_info(VLM_ID).sha  # Preserve this resolved commit for later reloads.
processor = AutoProcessor.from_pretrained(VLM_ID, revision=VLM_REVISION)
vlm = AutoModelForImageTextToText.from_pretrained(
    VLM_ID, revision=VLM_REVISION, dtype=DTYPE, device_map={"": 0})
vlm.eval()
print("Resident VLM:", VLM_ID, VLM_REVISION)


#### `ask_vlm()`

In [ ]:
def ask_vlm(image, question, max_new_tokens=100):
    """Return only generated tokens, not the image/question chat prefix."""
    messages = [{"role": "user", "content": [{"type": "image", "image": image.convert("RGB")},
                                             {"type": "text", "text": question}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt").to(vlm.device, dtype=DTYPE)
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0, prompt_length:], skip_special_tokens=True).strip()


### ⚙️ retrieval + shared metrics · *given*

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss

EMBEDDER_ID = "BAAI/bge-small-en-v1.5"
EMBEDDER_REVISION = model_info(EMBEDDER_ID).sha
embedder = SentenceTransformer(EMBEDDER_ID, revision=EMBEDDER_REVISION, device="cpu")
BGE_PREFIX = "Represent this sentence for searching relevant passages: "


#### `build_index()`

In [ ]:
def build_index(texts):
    """One normalized finite embedding per nonempty text; exact cosine index."""
    if not texts or any(not isinstance(t, str) or not t.strip() for t in texts):
        raise ValueError("Supply nonempty passage texts")
    vecs = np.ascontiguousarray(embedder.encode(texts, normalize_embeddings=True), dtype=np.float32)
    if vecs.ndim != 2 or len(vecs) != len(texts) or not np.isfinite(vecs).all():
        raise ValueError("Invalid passage embeddings")
    index = faiss.IndexFlatIP(vecs.shape[1]); index.add(vecs)
    return index, vecs


#### `format_ok()`

In [ ]:
# Defined in the GIVEN support cell above; this stable cell position is retained.


#### `bench()`

In [ ]:
# Defined in the GIVEN support cell above; this stable cell position is retained.


### ⚙️ the cross-stage contract · *given*

In [ ]:
# The support cell defines make_input(), split_input(), REQUIRED_SECTIONS and REFUSAL_PHRASES.
# Keep these contracts unchanged in learner implementations.


#### `make_input()`

In [ ]:
# Defined in the GIVEN support cell above; this stable cell position is retained.


#### `split_input()`

In [ ]:
# Defined in the GIVEN support cell above; this stable cell position is retained.


In [ ]:
# ⚙️ GIVEN · CAPSTONE_DOCS — the official internal documents of the RiskLens world ------
# Your Stage-2 corpus MUST include these four, verbatim, with these exact source ids:
# they are the product world your evidence engine must master. Add your own on top (D2.1).
CAPSTONE_DOCS = [
    {"source": "spec_kettle", "kind": "doc", "text":
        "KT-2000 electric kettle. Capacity 1.7 L. Body: borosilicate glass. "
        "Lid assembly: polypropylene, part PP-114. Warranty: 24 months."},
    {"source": "spec_toaster", "kind": "doc", "text":
        "TS-400 two-slot toaster. Lever mechanism part SP-77C revised in 2023-06 "
        "to fix reported sticking. Warranty: 12 months."},
    {"source": "spec_blender", "kind": "doc", "text":
        "BL-900 blender. Motor 900 W. Jar 1.5 L with silicone lid seal, part LS-22. "
        "Jar is dishwasher safe up to 60 degrees. Warranty: 12 months on motor."},
    {"source": "policy_returns", "kind": "doc", "text":
        "Return policy for all products (KT-2000, TS-400, BL-900): defective units "
        "returnable within 30 days for full refund; after 30 days warranty claims are "
        "repair or replacement. Water damage and misuse are excluded."},
]
print(len(CAPSTONE_DOCS), "official docs — include them in your corpus unchanged")

---
# STAGE 1 · VLM Defect Inspector  `(Module 1 · competency C1)`

**Mission:** you inherit `inspect_v0` — it *runs*, and it's *bad*, with **three planted bugs**.
Diagnose them, build a proper `inspect()`, and prove yours is better on a split you froze first.

| # | Deliverable | Where |
|---|---|---|
| D1.1 | Frozen split: ≥40 test images (≥20/class) + dev set, seed printed, frozen before any prompt work | SPLIT cell |
| D1.2 | All 3 baseline bugs diagnosed: symptom → root cause → fix, one markdown cell each | scaffolds |
| D1.3 | `inspect()` honoring the JSON contract, handles malformed replies; VLM choice justified | scaffold |
| D1.4 | Prompt ladder: ≥3 measured iterations on dev, incl. one few-shot attempt | scaffold |
| D1.5 | Error analysis: ≥3 misclassified test images shown, with hypotheses | scaffold |
| D1.6 | Evaluation cell run, outputs saved — beats `inspect_v0` on defect **F1**; cost/1k recorded | 📊 cell |
| D1.7 | Resolution-budget experiment: dev accuracy + sec/image at ≥3 image sizes, with a recommendation | scaffold |

**Suggested week plan:** Day 1–2: D1.1 + D1.2 · Day 3–4: D1.3 + D1.4 · Day 5: D1.5 + D1.7 ·
Day 6: evaluation cell + checklist. *(The live session covers everything needed for D1.1–D1.4.)*

### ⚙️ data — downloads ONLY the requested category (~150 MB, not the 5 GB repo) · *given*

In [ ]:
from huggingface_hub import snapshot_download
from PIL import Image as PILImage
import glob

#### `load_mvtec()`

In [ ]:
def load_mvtec(category="bottle", n_good=30, n_defective=30):
    """Load RGB image/label pairs; reject empty or undersized downloaded selections."""
    if not re.fullmatch(r"[a-z_]+", category):
        raise ValueError("Use a simple MVTec category name")
    if any(type(n) is not int or n < 0 for n in [n_good, n_defective]):
        raise ValueError("Image counts must be nonnegative integers")
    root = snapshot_download("TheoM55/mvtec_anomaly_detection", repo_type="dataset",
        allow_patterns=[f"images/train/{category}/*", f"images/test/{category}/*"])
    good = sorted(glob.glob(f"{root}/images/train/{category}/good/*.png"))[:n_good]
    bad = sorted(p for p in glob.glob(f"{root}/images/test/{category}/*/*.png") if "/good/" not in p)[:n_defective]
    if len(good) < n_good or len(bad) < n_defective:
        raise ValueError("Insufficient images; inspect the dataset snapshot and category paths")
    result = []
    for paths, label in [(good, "good"), (bad, "defective")]:
        for path in paths:
            with PILImage.open(path) as opened:
                image = opened.convert("RGB")
            image.info["source_path"] = path
            result.append((image, label))
    return result

print("Image loader ready. Reserve demonstration images outside frozen evaluation membership.")


### ⚙️ the baseline you must beat. It runs. It is bad ON PURPOSE. · *given*

In [ ]:
# Three planted bugs — your D1.2 diagnoses all of them:
#   BUG-1 (parser):   look at the verdict line. What does it do to "No defects found."?
#   BUG-2 (prompt):   no role, no defect checklist, no rule about reflections/glare.
#   BUG-3 (contract): defect_type and description are never filled -> Stage 2 has nothing
#                     to retrieve with, Stage 3 has nothing to analyze. Downstream starves.

#### `inspect_v0()`

In [ ]:
def inspect_v0(image):
    """Baseline inspector — weak by design; see the planted bugs above."""
    answer = ask_vlm(image, "Is this defective?", max_new_tokens=30)
    return {"verdict": "DEFECTIVE" if "DEFECT" in answer.upper() else "GOOD",
            "defect_type": None, "description": ""}

# Demonstrate this only on development/reserved images AFTER freezing D1.1.


### D1.1 · Freeze before prompt iteration

Create at least 40 test images (at least 20 per class), a separate development set and printed seed/membership/counts. Use `validate_image_split()` and `freeze_record()` after creating them. Preserve dated version history; local cell position is insufficient.


In [ ]:
# D1.1 — learner work. The helper validates evidence, not your methodology.
test_images = []
dev_images = []
raise NotImplementedError("Build and freeze D1.1, then remove this intentional scaffold stop")
# split_payload = {"seed": SEED, **validate_image_split(test_images, dev_images)}
# freeze_record("evidence/split_v1.json", split_payload)
# print(json.dumps(split_payload, indent=2))


### D1.2 · Bug diagnoses — one cell per bug
Replace each placeholder below. A diagnosis has three parts: **symptom** (what you observed —
show a concrete input/output), **root cause** (why the code does that), **fix** (what your
`inspect()` does differently). One short paragraph each; evidence beats length.

**BUG-1 · (your title)** — *symptom:* … · *root cause:* … · *fix:* …

**BUG-2 · (your title)** — *symptom:* … · *root cause:* … · *fix:* …

**BUG-3 · (your title)** — *symptom:* … · *root cause:* … · *fix:* …

### D1.3 · Implement the inspector

Fix all three inherited defects. Request structured JSON, validate the required fields and explain your domain prompt and model choice. Malformed model replies must not cause an uncaught parsing error. Define a conservative documented fallback, retain the raw reply and flag that fallback; never silently turn a parsing failure into a confident GOOD result. Infrastructure failures should remain visible in the evidence.


In [ ]:
# D1.3 — your work goes here
def inspect(image):
    """Your VLM inspector — same signature as inspect_v0, all three bugs fixed.

    Handles malformed replies without an uncaught parsing error. Document the conservative
    fallback and retain raw output plus a fallback flag; do not hide infrastructure errors.
    Returns:
        {"verdict": "GOOD"|"DEFECTIVE", "defect_type": str|None, "description": str}
    """
    raise NotImplementedError("D1.3 — implement, then delete this line")

### ✏️ D1.4 — prompt ladder — >=3 measured iterations on DEV

> The ladder (add one cell per rung below — keep every measurement visible):
> rung 1: your v1 prompt               -> dev accuracy
> rung 2: your improved v2             -> dev accuracy (what changed? why?)
> rung 3: few-shot with example images -> dev accuracy + a per-call token-cost note
> Conclude with ONE line: which rung inspect() uses, and why.

In [ ]:
# D1.4 — your work goes here
def dev_accuracy(predict_fn):
    """Fraction of dev_images classified correctly by predict_fn(image) -> "good"|"defective"."""
    return sum(predict_fn(img) == truth for img, truth in dev_images) / len(dev_images)

raise NotImplementedError("D1.4 — run the ladder, then delete this line")

### D1.5 · Analyze errors after the saved evaluation

Analyze at least three real errors with hypotheses. If there are fewer, include all errors and enough explicitly labeled borderline cases to total three. Do not invent failures or tune against these final-test results.


In [ ]:
# D1.5 — add your written error/borderline-case analysis below after D1.6.
# On final Run all, display the preserved analysis and genuine saved outputs.


### ✏️ D1.7 — resolution-budget experiment

> Image tokens dominate VLM cost. Measure the trade-off on DEV at >=3 sizes:
> for size in [256, 512, 1024]:
> resized_dev = [(img.resize((size, size)), y) for img, y in dev_images]
> -> dev accuracy + mean sec/image at each size
> Deliverable: a small printed table + ONE sentence: the size you'd run in production, why.

In [ ]:
# D1.7 — your work goes here
raise NotImplementedError("D1.7 — run the experiment, then delete this line")

### 📊 Evaluation · Stage 1 — *run when the stage is done; the saved output is what graders read*

In [ ]:
# D1.6 · Evidence generation, not automatic competency grading.
split_payload = {"seed": SEED, **validate_image_split(test_images, dev_images)}
freeze_record("evidence/split_v1.json", split_payload)

def _score_inspector(fn):
    records = []
    for image, truth in test_images:
        torch.cuda.synchronize(); started = time.perf_counter()
        try:
            note = fn(image)
            predicted = {"GOOD": "good", "DEFECTIVE": "defective"}.get(note.get("verdict"), "invalid")
            record = {"note": note, "prediction": predicted}
        except Exception as error:
            record = {"prediction": "invalid", "error": repr(error)}
        torch.cuda.synchronize()
        records.append({"image_id": image_id(image), "truth": truth,
                        "seconds": time.perf_counter() - started, **record})
    return {"records": records, "metrics": inspection_metrics(records)}

stage1 = {"baseline": _score_inspector(inspect_v0), "final": _score_inspector(inspect)}
stage1["environment"] = environment
stage1["split_sha256"] = stable_hash(split_payload)
write_json("evidence/stage1_outputs.json", stage1)
_seconds = [r["seconds"] for r in stage1["final"]["records"]]
_mean = sum(_seconds) / len(_seconds)
_cost = cost_per_1000(_mean, GPU_PRICE_PER_H) if GPU_PRICE_PER_H is not None else None
print("Baseline:", stage1["baseline"]["metrics"])
print("Final:", stage1["final"]["metrics"])
print("Improved defect F1:", stage1["final"]["metrics"]["defect_f1"] > stage1["baseline"]["metrics"]["defect_f1"])
print("Review fallback/invalid records, three errors or borderline cases, and the separate resolution experiment.")
if _cost is None: print("Cost evidence incomplete: set an explicit hourly-price assumption.")
record_metrics("stage1_vlm", **stage1["final"]["metrics"],
    baseline_defect_f1=stage1["baseline"]["metrics"]["defect_f1"],
    mean_seconds_per_image=_mean, hourly_price=GPU_PRICE_PER_H, estimated_cost_per_1000=_cost)


**Module 1 submission checklist** — before sending your link:
- [ ] D1.1 split cell: seed + class counts in the saved output, frozen before prompt iteration with dated version evidence
- [ ] D1.2 three bug diagnoses with concrete evidence
- [ ] D1.4 ladder: every rung's dev number visible, few-shot attempted, decision stated
- [ ] D1.5 three real errors or explicitly labeled borderline cases · D1.7 resolution table + recommendation
- [ ] 📊 evaluation cell output saved (F1 beats baseline, cost logged)

---
# STAGE 2 · RAG Evidence Engine  `(Module 2 · competency C2)`

**Mission:** the inspector now names defects — connect them to evidence. You inherit
`retrieve_v0`, a naive retriever. Build a corpus worth searching, a gold set worth trusting,
and a retriever that beats the baseline **measurably**.

| # | Deliverable | Where |
|---|---|---|
| D2.1 | Corpus: ≥100 real reviews (provenance stated) + the 4 `CAPSTONE_DOCS` verbatim + ≥2 docs you author (planted facts, one cross-doc link) | CORPUS cell |
| D2.2 | Frozen gold set: ≥20 questions (≥3 multi-hop), authored **before** any indexing | GOLD cell |
| D2.3 | `retrieve()` beats `retrieve_v0` on recall@4 over the gold set | scaffold |
| D2.4 | `rag_answer()`: grounded, cites `[source_id]`s, refuses via a `REFUSAL_PHRASES` phrase; ≥4/5 adversarial questions refused | scaffold |
| D2.5 | Faithfulness + answer-relevance scored (LLM judge or `ragas`), judge caveats stated | scaffold |
| D2.6 | Evaluation cell run, outputs saved — recall table baseline-vs-yours + refusal rate | 📊 cell |
| D2.7 | One measured retrieval upgrade beyond the prefix (chunking / reranker / hybrid / k-sweep): before/after recall table + verdict | scaffold |

**Suggested week plan:** Day 1–2: D2.1 + D2.2 · Day 3: D2.3 · Day 4: D2.4 + D2.5 ·
Day 5: D2.7 · Day 6: evaluation cell + checklist.

### ⚙️ real-review loader — streams the raw JSONL head, no legacy dataset script · *given*

#### `load_reviews()`

In [ ]:
def load_reviews(n=120, category_file="Appliances.jsonl", max_scanned=20000):
    """Bounded stream of real reviews with exact source revision and row provenance."""
    from datasets import load_dataset
    from huggingface_hub import HfApi
    import itertools
    if type(n) is not int or n<1 or type(max_scanned) is not int or max_scanned<n:
        raise ValueError("Positive n and max_scanned >= n required")
    if not isinstance(category_file,str) or "/" in category_file or not category_file.endswith(".jsonl"):
        raise ValueError("Provide a category JSONL filename")
    dataset="McAuley-Lab/Amazon-Reviews-2023"
    revision=HfApi().dataset_info(dataset).sha
    url=f"https://huggingface.co/datasets/{dataset}/resolve/{revision}/raw/review_categories/{category_file}"
    stream=load_dataset("json",data_files=url,split="train",streaming=True)
    rows=[]
    for source_row,ex in enumerate(itertools.islice(stream,max_scanned)):
        text=ex.get("text") or ""
        if isinstance(text,str) and 200<len(text.strip())<800:
            rows.append({"source":f"review_{len(rows):03d}","text":text.strip(),"kind":"review",
                "provenance":{"dataset":dataset,"revision":revision,"url":url,"source_row":source_row,
                    "parent_asin":ex.get("parent_asin"),"timestamp":ex.get("timestamp")}})
        if len(rows)==n:break
    if len(rows)<n:raise RuntimeError(f"Only {len(rows)} eligible rows in bounded scan; inspect the source")
    return rows
print("Review loader ready; save the returned snapshot and provenance before indexing.")


### ✏️ D2.1 — CORPUS  +  D2.2 · GOLD — gold is authored BEFORE any index exists

> corpus rows: {"source": str, "text": str, "kind": "review"|"doc"}
> requirements: >=100 reviews + the 4 CAPSTONE_DOCS verbatim + >=2 docs you author.
> Your authored docs need planted, testable facts and ONE fact that links two documents
> (that link becomes a multi-hop gold question).
> gold rows: {"q": str, "sources": [ids that MUST be retrieved], "ref": distinctive substring,
> "multihop": True on >=3 rows}
> requirements: >=20 questions, authored by READING the corpus — not by testing retrieval.

In [ ]:
# D2.1 — your work goes here
corpus = []

gold = []

raise NotImplementedError("D2.1 + D2.2 — build corpus & gold set, then delete this line")

### ⚙️ the baseline retriever you must beat (builds on YOUR corpus, lazily) · *given*

In [ ]:
_ix_v0 = None

#### `retrieve_v0()`

In [ ]:
baseline_index, baseline_vectors = build_index([d["text"] for d in corpus])

def retrieve_v0(query, k=4):
    """Deliberately naive unprefixed dense retrieval; safe top-k bounds."""
    if type(k) is not int or k < 1: raise ValueError("k must be a positive integer")
    if not corpus: raise ValueError("Build the frozen corpus first")
    query_vector = np.ascontiguousarray(embedder.encode([query], normalize_embeddings=True), dtype=np.float32)
    scores, indices = baseline_index.search(query_vector, min(k, len(corpus)))
    return [{**corpus[int(i)], "score": float(score)} for score, i in zip(scores[0], indices[0]) if i >= 0]


### ✏️ D2.3 — your retriever



In [ ]:
# D2.3 — your work goes here
def retrieve(query, k=4):
    """Your retriever. Returns top-k rows: [{"source", "text", "kind", "score"}, ...].

    Minimum bar: the BGE query prefix. Where the real recall@4 gains come from is your
    D2.7 experiment — chunking, a reranker, hybrid keyword+dense, or a k/score cutoff.
    """
    raise NotImplementedError("D2.3 — implement, then delete this line")

### D2.4 · Grounded answering and five preselected unsupported questions

Freeze exactly five genuinely unanswerable questions before testing. At least four must receive a meaningful refusal containing a specified phrase, without a fabricated continuation. Report any extra adversarial cases separately.


In [ ]:
# D2.4 — your work goes here
def rag_answer(question, k=4):
    """Retrieve, then answer FROM the retrieved text only.

    Returns:
        (answer_text, hits) — hits is the retrieve() output used, so citations are auditable.
    Prompt requirements: context block first, RULES second (answer only from context; cite
    [source_id] after each claim; refuse with a REFUSAL_PHRASES phrase when the context
    doesn't cover it), question last.
    """
    raise NotImplementedError("D2.4 — implement, then delete this line")

ADVERSARIAL = []

### ✏️ D2.5 — faithfulness + answer-relevance

> Score BOTH axes over the gold set — they are independent failure modes:
> faithfulness: is every claim supported by the retrieved context?
> relevance:    does the answer address the question asked?
> Use an LLM judge (state which model) or the `ragas` library. REQUIRED: one markdown
> sentence on judge bias (what does sharing weights with the generator do to the score?).

In [ ]:
# D2.5 — your work goes here
raise NotImplementedError("D2.5 — score both axes, then delete this line")

### ✏️ D2.7 — one measured retrieval upgrade

> Pick ONE lever beyond the prefix: chunking strategy, a cross-encoder reranker, hybrid
> BM25+dense, or a k/score-threshold sweep. Show the SAME recall table before and after,
> and end with one sentence: adopted or rejected, and why.

In [ ]:
# D2.7 — your work goes here
raise NotImplementedError("D2.7 — run the experiment, then delete this line")

### 📊 Evaluation · Stage 2 — *run when the stage is done; the saved output is what graders read*

In [ ]:
# D2.6 · Keep full hits/answers. Phrase and substring probes are not faithfulness scores.
validate_corpus_gold(corpus, gold, CAPSTONE_DOCS)
if len(ADVERSARIAL) != 5 or len(set(ADVERSARIAL)) != 5:
    raise ValueError("Preselect exactly five distinct unsupported questions; extras are a separate experiment")
freeze_record("evidence/rag_gold_v1.json", {"corpus": corpus, "gold": gold})
freeze_record("evidence/refusal_cases_v1.json", ADVERSARIAL)
baseline_recall, baseline_hits = recall_table(retrieve_v0, gold)
final_recall, final_hits = recall_table(retrieve, gold)
answer_records = []
for group, questions in [("gold", [g["q"] for g in gold]), ("unsupported", ADVERSARIAL)]:
    for question in questions:
        answer, hits = rag_answer(question)
        answer_records.append({"group": group, "question": question, "answer": answer,
            "hits": hits, **refusal_diagnostic(answer, hits)})
stage2 = {"baseline": baseline_recall, "final": final_recall,
    "baseline_hits": baseline_hits, "final_hits": final_hits, "answers": answer_records}
write_json("evidence/stage2_outputs.json", stage2)
print("Baseline:", baseline_recall, "Final:", final_recall)
print("Improved recall@4:", final_recall["recall@4"] > baseline_recall["recall@4"])
for row in answer_records: print(json.dumps(row, ensure_ascii=False))
print("Complete D2.5's separate faithfulness/relevance evaluation and human citation/refusal review.")
record_metrics("stage2_rag", n_corpus=len(corpus), n_gold=len(gold),
    baseline_recall=baseline_recall, final_recall=final_recall,
    refusal_phrase_matches=sum(r["phrase_match"] for r in answer_records if r["group"] == "unsupported"),
    meaningful_refusals=None, generation_review="PENDING — record actual D2.5 evidence")


**Module 2 submission checklist:**
- [ ] D2.1 provenance stated; authored docs have planted facts + one cross-doc link
- [ ] D2.2 gold set frozen before indexing with a dated version record
- [ ] D2.5 both judge axes scored, judge-bias sentence present
- [ ] D2.7 before/after table + adopt/reject verdict
- [ ] 📊 evaluation cell output saved

---
# STAGE 3 · Fine-Tuned Analyst  `(Module 3 · competency C3)`

**Mission:** no baseline this week — the base model IS the baseline. Build a real teacher-model
dataset, prove it's clean, train QLoRA, compare base and tuned behavior three ways; no minimum tuning win-rate is required, and ablate one knob.

| # | Deliverable | Where |
|---|---|---|
| D3.1 | ≥150 (input → report) pairs from a **real teacher model** (the template generator from the ungraded assignment is not accepted); inputs use `make_input()` | scaffold |
| D3.2 | Dataset card complete; dedup + decontamination shown as *actions* with counts | CARD cell + scaffold |
| D3.3 | Frozen `analyst_test_rows` ≥25 + declared similarity screen (max train↔test cosine ≤ 0.90) | CARD cell |
| D3.4 | QLoRA fine-tune (train-or-load pattern); adapter saved to `analyst-lora-adapter/` | scaffold |
| D3.5 | Three-way eval: format adherence · rubric score (stronger judge, caveats) · blind **randomized** win-rate vs base | scaffold |
| D3.6 | Evaluation cell run, outputs saved; `WEEK3_RESULTS` filled from your eval | 📊 cell |
| D3.7 | Ablation: change ONE training knob (r, epochs, LR, or data size), retrain, compare on the frozen set; one-paragraph conclusion | scaffold |

**Suggested week plan:** Day 1–2: D3.1 (teacher generation is the slow part — start early) ·
Day 3: D3.2 + D3.3 · Day 4: D3.4 + D3.5 · Day 5: D3.7 · Day 6: evaluation cell + checklist.

### ✏️ D3.2 — D3.3 · DATASET CARD + frozen test rows — complete BEFORE training



In [ ]:
# D3.2 — your work goes here
DATASET_CARD = {
    "teacher_model": None,
    "teacher_revision_and_settings": None,
    "embedding_model_revision_and_method": None,
    "frozen_split_record": None,             # exact model/API that wrote the reports
    "generation_strategy": None,       # how you varied products/defects/severities/phrasing
    "n_raw": None,                     # pairs before cleaning (>=150)
    "dedup_threshold": None,           # cosine threshold used (0.90 recommended)
    "n_after_dedup": None,
    "n_dropped_decontamination": None, # train rows DROPPED for sitting too close to test rows
    "max_train_test_similarity": None, # the declared similarity screen (must be <= 0.90)
    "n_train": None,
    "n_test": None,                    # >= 25
}

analyst_test_rows = []                 # >=25 frozen {"input": make_input(...), "output": report}

WEEK3_RESULTS = {                      # populate from complete saved D3.5 judgments; counts alone are not proof
    "rubric_score_tuned": None,        # mean 1-5 from the judge
    "rubric_score_base": None,
    "win_rate": None,                  # e.g. "17/25" — report the count, not just a percent
    "wins": None, "losses": None, "ties": None, "invalid_judgments": None,
    "denominator_and_order_protocol": None,
    "judge_model": None,               # must be stronger than / different from the generator
}

raise NotImplementedError("D3.2 + D3.3 — build the dataset + card, then delete this line")

### ✏️ D3.1 — teacher generation  +  D3.2 · hygiene actions

> Workspace for: calling your teacher model, assembling raw pairs, then the hygiene chain
> from the Week-3 assignment — dedup (embedding cosine), split, decontaminate (DROP leaky
> train rows). Print counts at every step; those numbers fill DATASET_CARD.

In [ ]:
# D3.1 — your work goes here
raise NotImplementedError("D3.1 + D3.2 — generate and clean, then delete this line")

### D3.4 · Train deliberately; load on delivery

Use explicit TRAIN / LOAD modes. TRAIN creates a new named run and saves base ID/revision, tokenizer, adapter configuration, parameters, loss history and the artifact. LOAD restores those weights and checks their identity; it must not silently start training if files are missing. Demonstrate a fresh reload. Set `model.eval()` to disable dropout and set `model.config.use_cache=True` separately for generation. Preserve the same data and runtime choices when comparing.


In [ ]:
# D3.4 — your work goes here
def analyst_report(defect_note, evidence):
    """Your fine-tuned Analyst: (note, evidence) -> a report following REQUIRED_SECTIONS."""
    raise NotImplementedError("D3.4 — implement after training, then delete this line")

raise NotImplementedError("D3.4 — train or load, then delete this line")

### ✏️ D3.5 — three-way eval on the frozen rows

> 1) format adherence: format_ok() over your generations (deterministic, no judge)
> 2) rubric score:     a judge grades 1-5 vs the reference outputs — judge must differ from
> the generator; state its name + the bias caveat
> 3) win-rate:         blind pairwise tuned-vs-base with RANDOMIZED A/B position
> (base outputs via `with model.disable_adapter():` — chat(base,...) lies)
> Fill WEEK3_RESULTS above from these numbers.

In [ ]:
# D3.5 — your work goes here
raise NotImplementedError("D3.5 — run all three, then delete this line")

### ✏️ D3.7 — ablation — one knob, one honest comparison

> Change exactly ONE of: LoRA r · epochs · learning rate · training-set size. Retrain to a
> second adapter dir, re-run the SAME frozen-set eval, and write one paragraph: what moved,
> what didn't, and what you'd pick for production. Keep both adapters' numbers visible.

In [ ]:
# D3.7 — your work goes here
raise NotImplementedError("D3.7 — ablate, then delete this line")

### 📊 Evaluation · Stage 3 — *run when the stage is done; the saved output is what graders read*

In [ ]:
# D3.6 · Preserve all judge identities and count strings, not just numeric fields.
if any(v is None for v in DATASET_CARD.values()): raise ValueError("Complete every dataset-card field")
if DATASET_CARD["n_raw"] < 150 or len(analyst_test_rows) < 25:
    raise ValueError("Require at least 150 raw real-teacher pairs and 25 frozen tests")
similarity = DATASET_CARD["max_train_test_similarity"]
if not math.isfinite(similarity) or not -1 <= similarity <= .90:
    raise ValueError("Declare the method and show finite max train/test cosine <= .90")
if any(v is None for v in WEEK3_RESULTS.values()): raise ValueError("Populate WEEK3_RESULTS from saved judgments")
stage3_outputs = [{"input": r["input"], "reference": r["output"],
                   "output": analyst_report(*split_input(r))} for r in analyst_test_rows]
stage3_format = sum(format_ok(r["output"]) for r in stage3_outputs) / len(stage3_outputs)
write_json("evidence/stage3_outputs.json", {"rows": stage3_outputs,
    "dataset_card": DATASET_CARD, "comparison": WEEK3_RESULTS})
record_metrics("stage3_analyst", n_test=len(analyst_test_rows), format_adherence=stage3_format,
    dataset_card=DATASET_CARD, comparison=WEEK3_RESULTS)
print("Format rate:", stage3_format, "— content and blind A/B evidence must remain separately visible.")


**Module 3 checkpoint:** preserve the real teacher calls, dataset card and cleaning counts, frozen tests, trained/reloaded adapter, format/content/blind A/B outputs and both ablation runs. State a concrete expected out-of-distribution failure. All C3 criteria are required; no improvement margin is invented.

# Module 4 · Optimize the trained Analyst — C4

Complete D4.1–D4.4 here: technique note, predeclared gate, two compressed trained-model arms, reference and half-precision footprints, same-row format/content comparison, measured runtime/cost and an accept/reject/fallback decision. Keep the Stage 3 reference and all failed candidates visible. Integration follows in Module 5.


### ✏️ D4.1 — GATE — commit BEFORE measuring; do not touch after the re-score



In [ ]:
# D4.1 — complete and preserve BEFORE compressed-model scoring.
GATE = {"max_format_drop_pts": None, "max_content_drop_pts": None,
        "format_units": "percentage points", "content_units": "cosine difference multiplied by 100",
        "reference": None, "candidates": [], "gpu_price_per_h": GPU_PRICE_PER_H,
        "arm_to_ship": None, "rationale": None}
raise NotImplementedError("Declare the gate, arm identities, price and rationale, then freeze the record")
# freeze_record("evidence/gate_v1.json", GATE)


### D4.2 · Two compressed arms of your trained Analyst

Measure at least two compressed versions, such as folded int8 and folded NF4, and the half-precision footprint denominator. Keep the original Stage 3 reference. Record adapter/fold lineage and actual footprints under one definition; an unrelated quantized base is not your trained Analyst.


In [ ]:
# D4.2 — your work goes here
raise NotImplementedError("D4.2 — build the arms, then delete this line")

### ✏️ D4.3 — the shipped-arm contract



In [ ]:
# D4.3 — your work goes here
def analyst_report_compressed(defect_note, evidence):
    """Same contract as analyst_report, served by GATE["arm_to_ship"]."""
    raise NotImplementedError("D4.3 — implement, then delete this line")

### D4.4 · Actual serving measurements

Measure the reference and every candidate: warm-ups, run count, input/output lengths, p50/p95, actual newly generated token IDs per elapsed second, memory and cost per 1,000 reports with stated price/utilization. Synchronize CUDA around timing. Use mean elapsed work for a sequential cost estimate; p50 is a separately labeled proxy. Small-sample p95 is not a production SLA. Explain observed trade-offs without assuming low-bit models are faster.


In [ ]:
# D4.4 — your work goes here
raise NotImplementedError("D4.4 — measure, then delete this line")

### Module 4 standalone checkpoint · required before integration

Below D4.4, save a table for the Stage 3 reference, half-precision footprint reference and at least two compressed trained-Analyst arms. For each arm show identity, footprint/ratio, unrounded format rate, content-cosine proxy, drops under the frozen gate, verdict and benchmark evidence. Re-score the reference on the same rows. Use `quality_gate()` for consistent units; do not reinterpret cosine as factual accuracy.

Record the decision and feasible fallback with its limitations. Save these outputs without calling `analyze_product()`. This checkpoint completes the Module 4 evidence even though the original combined Stage 4 check appears later.


In [ ]:
# Add your genuine standalone Module 4 checkpoint outputs here.
# Do not use an integration stub to satisfy this checkpoint.


# Module 5 · Integrate, evaluate and explain — C5

Continue with D4.5–D4.7; the D labels are retained for compatibility. Connect real components, preserve the GOOD short-circuit, and demonstrate one good plus two defective products. Save a batch of at least ten products, end-to-end and stage timings, a failure taxonomy and the deployment note. Complete the clean run/load, reflection and written walkthrough. C5.1 is reviewed first.


### ✏️ D4.5 — the assembled pipeline

> Demo on >=3 products (>=1 good, >=2 defective) in cells below — full outputs saved.

In [ ]:
# D4.5 — your work goes here
def analyze_product(image):
    """The full RiskLens pipeline: photo in, risk report out.

    Stage 1: inspect(image)                  -> defect note dict (GOOD short-circuits)
    Stage 2: retrieve(defect-based query)    -> evidence rows -> "- source: text" bullets
    Stage 3: analyst_report_compressed(...)  -> the report. The note goes in as PROSE —
             the shape it was trained on (e.g. f"{defect_type}: {description}").
    Returns:
        {"note": dict, "evidence": str, "report": str}
    """
    raise NotImplementedError("D4.5 — compose, then delete this line")

### ✏️ D4.7 — batch stress test

> Run analyze_product over >=10 test images end-to-end. Deliver:
> 1. end-to-end p50/p95 per product
> 2. stage-time breakdown (VLM vs retrieval vs Analyst — where does the time go?)
> 3. a failure taxonomy: for every product whose report is wrong/weak, WHICH stage broke?
> (this is why the stages have separate evals — prove it with your own numbers)

In [ ]:
# D4.7 — your work goes here
raise NotImplementedError("D4.7 — stress it, then delete this line")

### D4.6 · Deployment note *(replace this cell's text)*
Where would RiskLens run in production (GGUF/CPU? vLLM on GPU? a Space?), what breaks first at
100× traffic (name the stage and the resource), and the ONE monitoring signal you'd page on.

### 📊 Evaluation · Stage 4 — *run when the stage is done; the saved output is what graders read*

In [ ]:
# Combined Stage 4 / Module 5 check — run only after real integration.
# D4.1–D4.4 standalone quality/benchmark evidence was already saved in Module 4.
demo = analyze_product(test_images[0][0])
if not isinstance(demo, dict) or not {"note", "evidence", "report"} <= set(demo):
    raise ValueError("Pipeline must return note, evidence and report")
if not isinstance(demo["note"], dict) or not all(isinstance(demo[k], str) for k in ["evidence", "report"]):
    raise TypeError("Required pipeline types: dict, str, str")
write_json("evidence/integration_contract_demo.json", demo)
print(json.dumps(demo, indent=2))
print("Retain all three demonstrations and the >=10-product batch with stage timing/failure review.")
record_metrics("stage4_ship", selected_arm=GATE["arm_to_ship"],
               integration_contract_checked=True, full_batch_evidence="See D4.7 saved outputs",
               competency_judgment="Human review required")


# Final delivery · Reflection and written walkthrough

Provide a clean run/load path, accessible weights/data, environment versions, full ledger and preserved experiment history. D4.6 names the runtime, 100-times-traffic bottleneck and one actionable monitoring signal.

Write the separate ≤300-word reflection below. Then add a written walkthrough: explain one actual component and the effect of removing a named part; trace a demonstrated product; defend a reported metric; analyze a failed experiment or unsupported claim. Separately issued exam evidence may replace equivalent explanation evidence. No separate recording or paid hosting is required.


### Final saved summary


In [ ]:
FINAL_REFLECTION = """Replace with your reflection, at most 300 words."""
if len(FINAL_REFLECTION.split()) > 300: raise ValueError("Reflection exceeds 300 words")
show_ledger()
print("Retain capstone_metrics_history.jsonl and full experiment outputs with your dated submission snapshot.")
